# College Scorecard Provisional Analysis (DOE) 

Hypothesis 1 (College Scorecard): Students who attend more racially and demographically diverse universities will demonstrate higher median earnings after graduation, reflecting the broad economic and social benefits of diverse educational environments.

- Expect that more homogenous universities (excl. HBCU's) will have worse outcomes for students
- Control for: family income distribution, completion rates


In [29]:
#setup chunk
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
from IPython.display import IFrame

In [81]:
#data cleaning process-- especially making smaller datasets that are more manageable and only include relevant variables

dat = pd.read_csv('data/Glynn_data/combined_data.csv')
dat.columns

Index(['univ_id', 'univ_name', 'state', 'zip_code', 'hbcu', 'pbi', 'aanapii',
       'hsi', 'tribal', 'percent_pell', 'completion_rate_fry', 'ugds_white',
       'ugds_black', 'ugds_hisp', 'ugds_asian', 'ugds_aian', 'ugds_nhpi',
       'ugds_2mor', 'ugds_nra', 'ugds_unkn', 'ugds_whitenh', 'ugds_blacknh',
       'ugds_api', 'ugds_aianold', 'ugds_hispold', 'ugds_men', 'ugds_women',
       'md_earn_wne_p10', 'md_earn_wne_p6', 'md_earn_wne_p8',
       'md_earn_wne_inc1_p6', 'md_earn_wne_inc2_p6', 'md_earn_wne_inc3_p6',
       'md_earn_wne_indep1_p6', 'md_earn_wne_indep0_p6',
       'md_earn_wne_male0_p6', 'md_earn_wne_male1_p6', 'md_earn_wne_inc1_p8',
       'md_earn_wne_inc2_p8', 'md_earn_wne_inc3_p8', 'md_earn_wne_indep1_p8',
       'md_earn_wne_indep0_p8', 'md_earn_wne_male0_p8', 'md_earn_wne_male1_p8',
       'md_earn_wne_inc1_p10', 'md_earn_wne_inc2_p10', 'md_earn_wne_inc3_p10',
       'md_earn_wne_indep1_p10', 'md_earn_wne_indep0_p10',
       'md_earn_wne_male0_p10', 'md_earn_wne_mal

## variable descriptions: 

- ugds_: different racial/ demographic variables of the student body at a university 
- MD_earn_: Median earnings are calculated for individuals that were federally aided, were working, and were not enrolled in school as of the measurement point. Earnings were measured based on wages and deferred compensation reported via IRS form W-2 plus positive self-employment earnings reported via Schedule SE. Individuals with $1 or more of earnings were considered to be working and were included in the median earnings calculation.
- 

In [85]:
#other dataset features
dat.info()
dat['univ_id'].nunique()

<class 'pandas.DataFrame'>
RangeIndex: 13027 entries, 0 to 13026
Data columns (total 84 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   univ_id                 13027 non-null  int64  
 1   univ_name               13027 non-null  str    
 2   state                   13027 non-null  str    
 3   zip_code                13027 non-null  str    
 4   hbcu                    0 non-null      float64
 5   pbi                     0 non-null      float64
 6   aanapii                 0 non-null      float64
 7   hsi                     0 non-null      float64
 8   tribal                  0 non-null      float64
 9   percent_pell            11409 non-null  float64
 10  completion_rate_fry     4544 non-null   float64
 11  ugds_white              11485 non-null  float64
 12  ugds_black              11485 non-null  float64
 13  ugds_hisp               11485 non-null  float64
 14  ugds_asian              11485 non-null  float64
 

6678

In [89]:
#need to find where the university ID is equal to one again at the bottom half of the dataset becasuse I forgot to specify whether 
    #an entry was for 2021 or 2022 when I merged the datasets then deleted the files. 
#find the very first entry
dat['univ_id'].iloc[0]

np.int64(100654)

In [93]:
#check where the dataset restarts by looking at which row number has this entry a second time
dat[dat['univ_id']== 100654] 

,univ_id,univ_name,state,zip_code,hbcu,pbi,aanapii,hsi,tribal,percent_pell,...,md_earn_wne_indep0_p11,md_earn_wne_indep1_p11,md_earn_wne_male0_p11,md_earn_wne_male1_p11,inc_pct_lo,inc_pct_m1,inc_pct_m2,inc_pct_h1,inc_pct_h2,inc_n
0,100654,Alabama A & M University,AL,35762,NaN,NaN,NaN,NaN,NaN,0.6853,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6543,100654,Alabama A & M University,AL,35762,NaN,NaN,NaN,NaN,NaN,0.6536,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [100]:
#making sure that this is actually a restarted entry: 
dat[6540:6550] #no, the first dataset seems to cut off around the letter P before going back to A&M

,univ_id,univ_name,state,zip_code,hbcu,pbi,aanapii,hsi,tribal,percent_pell,...,md_earn_wne_indep0_p11,md_earn_wne_indep1_p11,md_earn_wne_male0_p11,md_earn_wne_male1_p11,inc_pct_lo,inc_pct_m1,inc_pct_m2,inc_pct_h1,inc_pct_h2,inc_n
6540,49576722,Pennsylvania State University-Penn State Harri...,PA,17057-4846,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6541,49576723,Pennsylvania State University-Penn State Brand...,PA,19063-5522,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6542,49664501,Burlington County Institute of Technology - Ad...,NJ,080550000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6543,100654,Alabama A & M University,AL,35762,NaN,NaN,NaN,NaN,NaN,0.6536,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6544,100663,University of Alabama at Birmingham,AL,35294-0110,NaN,NaN,NaN,NaN,NaN,0.3308,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6545,100690,Amridge University,AL,36117-3553,NaN,NaN,NaN,NaN,NaN,0.7769,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6546,100706,University of Alabama in Huntsville,AL,35899,NaN,NaN,NaN,NaN,NaN,0.2173,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6547,100724,Alabama State University,AL,36104-0271,NaN,NaN,NaN,NaN,NaN,0.6976,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6548,100751,The University of Alabama,AL,35487-0100,NaN,NaN,NaN,NaN,NaN,0.1788,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6549,100760,Central Alabama Community College,AL,35010,NaN,NaN,NaN,NaN,NaN,0.2985,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [99]:
dat[13024:13026] #but maricopa is an entry twice... so the original dataset must be complete, just out of order? 
dat[dat['univ_id']== 49501302] 

,univ_id,univ_name,state,zip_code,hbcu,pbi,aanapii,hsi,tribal,percent_pell,...,md_earn_wne_indep0_p11,md_earn_wne_indep1_p11,md_earn_wne_male0_p11,md_earn_wne_male1_p11,inc_pct_lo,inc_pct_m1,inc_pct_m2,inc_pct_h1,inc_pct_h2,inc_n
6518,49501302,Western Maricopa Education Center - Northeast ...,AZ,85027-0000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13025,49501302,Western Maricopa Education Center - Northeast ...,AZ,85027-0000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#assign 2021/ 2022 to half of the dataset and fix the above later. 
dat['year'] = 2021
restart = dat['univ_id']

In [ ]:
#initial exploratory analysis: Hypothesis 1 (College Scorecard): Students who attend more racially and demographically diverse universities will 
    #demonstrate higher median earnings after graduation, reflecting the broad economic and social benefits of diverse educational environments.
#Expect that more homogenous universities (excl. HBCU’s) will have worse outcomes for students
#plot the relationship between earnings and percent of diff demographic gorups on campus


In [ ]:
#Control for: family income distribution, completion rates